# E2 BioBART Sentence No-context Fine-tuning

CLEF SimpleText Task 1.1 sentence-level biomedical text simplification.

This notebook fine-tunes a biomedical encoder-decoder model directly on sentence pairs without using any document or neighbouring-sentence context.

## Objective

Experiment E2 tests whether a biomedical-domain encoder-decoder model can improve sentence simplification quality compared with the previous Llama zero-shot and FLAN-T5 fine-tuning baselines.

## Imports and Setup

The notebook uses PyTorch through Hugging Face Transformers and Datasets. It does not install or change the PyTorch stack from inside the notebook, which keeps CUDA environments such as RunPod stable.

In [ ]:
from __future__ import annotations

import gc
import os
import random
import time
from collections import Counter
from pathlib import Path
from typing import Any

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))

import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    import evaluate
except ImportError:
    evaluate = None

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())

## Configuration

The preferred model is `GanjinZero/biobart-v2-base`. If it is unavailable from Hugging Face in the current environment, the loading cell falls back to `GanjinZero/biobart-base`, then `facebook/bart-base`.

In [ ]:
SEED = 42
MODEL_NAME = "GanjinZero/biobart-v2-base"
MODEL_CANDIDATES = [
    MODEL_NAME,
    "GanjinZero/biobart-base",
    "facebook/bart-base",
]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sentence_no_context"
TRAIN_PATH = DATA_DIR / "train_clean.csv"
VAL_PATH = DATA_DIR / "val_clean.csv"
TEST_PATH = DATA_DIR / "test_clean.csv"

OUTPUT_DIR = PROJECT_ROOT / "models" / "biobart_sentence_no_context"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "biobart_sentence_no_context_predictions.csv"
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16_supported = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
fp16_enabled = False
bf16_enabled = bf16_supported

print(f"Project root: {PROJECT_ROOT}")
print(f"Preferred model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"fp16 enabled: {fp16_enabled}")
print(f"bf16 enabled: {bf16_enabled}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Prediction path: {PREDICTION_PATH}")

## Dataset

The experiment uses the preprocessed no-context sentence-level files. Only `complex` is used as model input and `simple` is used as the target. Metadata columns are kept for saving predictions but are not used as model features.

In [ ]:
REQUIRED_COLUMNS = ["pair_id", "sent_id", "label", "complex", "simple"]

def load_clean_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} file: {path}")
    df = pd.read_csv(path)
    missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {missing_columns}")
    df = df[REQUIRED_COLUMNS].copy()
    for column in ["complex", "simple"]:
        df[column] = df[column].fillna("").astype(str).str.strip()
    df = df[df["complex"].ne("") & df["simple"].ne("")].reset_index(drop=True)
    return df

train_df = load_clean_split(TRAIN_PATH, "train")
val_df = load_clean_split(VAL_PATH, "validation")
test_df = load_clean_split(TEST_PATH, "test")

print(f"Loaded train: {len(train_df):,} rows from {TRAIN_PATH.relative_to(PROJECT_ROOT)}")
print(f"Loaded validation: {len(val_df):,} rows from {VAL_PATH.relative_to(PROJECT_ROOT)}")
print(f"Loaded test: {len(test_df):,} rows from {TEST_PATH.relative_to(PROJECT_ROOT)}")
print("Columns:", train_df.columns.tolist())
print()
print("Missing values after cleaning:")
display(pd.DataFrame({
    "train": train_df.isna().sum(),
    "validation": val_df.isna().sum(),
    "test": test_df.isna().sum(),
}))

## Data Inspection

Word length statistics help confirm that the fixed token limits are reasonable for this sentence-level task.

In [ ]:
display(train_df.head())

def word_count(text: str) -> int:
    return len(str(text).split())

length_frames = []
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    length_frames.append(
        pd.DataFrame(
            {
                "split": split_name,
                "complex_words": df["complex"].map(word_count),
                "simple_words": df["simple"].map(word_count),
            }
        )
    )
length_df = pd.concat(length_frames, ignore_index=True)

length_stats = (
    length_df.groupby("split")[["complex_words", "simple_words"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)
display(length_stats)

print("Train label distribution:")
display(train_df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))

max_source_length = 256
max_target_length = 128
print(f"max_source_length = {max_source_length}")
print(f"max_target_length = {max_target_length}")

## Tokenization

Inputs are formatted as `Rewrite this biomedical sentence in simpler language: {complex}`. Targets are the reference simplifications. Padding token IDs in labels are replaced with `-100` so the loss ignores padding.

In [ ]:
INPUT_PREFIX = "Rewrite this biomedical sentence in simpler language: "

def load_tokenizer(model_candidates: list[str]) -> tuple[Any, str]:
    errors = []
    for candidate in model_candidates:
        try:
            tokenizer = AutoTokenizer.from_pretrained(candidate)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            print(f"Loaded tokenizer: {candidate}")
            return tokenizer, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load tokenizer {candidate}: {exc}")
    raise RuntimeError("Could not load any tokenizer candidate.\n" + "\n".join(errors))

tokenizer, RESOLVED_MODEL_NAME = load_tokenizer(MODEL_CANDIDATES)

def preprocess_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    inputs = [INPUT_PREFIX + str(text) for text in examples["complex"]]
    targets = [str(text) for text in examples["simple"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
    )["input_ids"]

    labels = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label]
        for label in labels
    ]
    model_inputs["labels"] = labels
    return model_inputs

sample_prompt = INPUT_PREFIX + train_df.loc[0, "complex"]
print(sample_prompt)
print("Target:", train_df.loc[0, "simple"])

## Dataset Creation

The pandas splits are converted to Hugging Face Datasets, tokenized in batches, and formatted as PyTorch tensors.

In [ ]:
train_dataset_raw = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset_raw = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset_raw = Dataset.from_pandas(test_df, preserve_index=False)

remove_columns = train_dataset_raw.column_names
train_dataset = train_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
val_dataset = val_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
test_dataset = test_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)

# Keep Python lists here. DataCollatorForSeq2Seq performs dynamic padding and tensor conversion.
# Pre-formatting as torch can trigger slow list-of-ndarray tensor warnings for labels.

print(train_dataset)
print(val_dataset)
print(test_dataset)

In [ ]:
def inspect_tokenized_labels(dataset: Dataset, name: str, n: int = 3) -> None:
    print(f"{name} label sanity check")
    for idx in range(min(n, len(dataset))):
        labels = dataset[idx]["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        active_labels = [token_id for token_id in labels if token_id != -100]
        print(f"  example {idx}: active target tokens = {len(active_labels)}")
        print("  decoded target:", tokenizer.decode(active_labels, skip_special_tokens=True))
    empty_count = 0
    for row in dataset:
        labels = row["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        if not any(token_id != -100 for token_id in labels):
            empty_count += 1
    print(f"  empty targets: {empty_count} / {len(dataset)}")
    if empty_count:
        raise ValueError(f"{name} has empty tokenized targets; check the simple column and preprocessing.")

inspect_tokenized_labels(train_dataset, "train")
inspect_tokenized_labels(val_dataset, "validation")

## Model Loading

BioBART and BART are encoder-decoder models. The same resolved model name is used for tokenizer and model weights. If the preferred BioBART checkpoint is not available, the notebook uses the configured fallback list.

In [ ]:
def load_seq2seq_model(model_candidates: list[str], resolved_tokenizer_model: str) -> tuple[Any, str]:
    ordered_candidates = [resolved_tokenizer_model] + [
        candidate for candidate in model_candidates if candidate != resolved_tokenizer_model
    ]
    errors = []
    for candidate in ordered_candidates:
        try:
            model = AutoModelForSeq2SeqLM.from_pretrained(candidate)
            print(f"Loaded model: {candidate}")
            return model, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load model {candidate}: {exc}")
    raise RuntimeError("Could not load any model candidate.\n" + "\n".join(errors))

model, RESOLVED_MODEL_NAME = load_seq2seq_model(MODEL_CANDIDATES, RESOLVED_MODEL_NAME)
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

print(f"Resolved model: {RESOLVED_MODEL_NAME}")
print(f"Model loaded on: {next(model.parameters()).device}")

## Training

The run uses `Seq2SeqTrainer` with early stopping. `fp16` is disabled by default because it can cause unstable `NaN` losses for encoder-decoder models. If GPU memory is insufficient, reduce `per_device_train_batch_size` to `4` and set `gradient_accumulation_steps = 2`.

In [ ]:
def build_training_args() -> Seq2SeqTrainingArguments:
    base_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=5,
        learning_rate=3e-5,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        fp16=fp16_enabled,
        bf16=bf16_enabled,
        logging_nan_inf_filter=False,
        report_to="none",
        seed=SEED,
    )
    try:
        return Seq2SeqTrainingArguments(
            evaluation_strategy="epoch",
            **base_kwargs,
        )
    except TypeError:
        return Seq2SeqTrainingArguments(
            eval_strategy="epoch",
            **base_kwargs,
        )

training_args = build_training_args()

def build_trainer() -> Seq2SeqTrainer:
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        return Seq2SeqTrainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        return Seq2SeqTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer = build_trainer()
trainer.train()
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))
print(f"Saved best model to: {BEST_MODEL_DIR.relative_to(PROJECT_ROOT)}")

## Training Diagnostics

The log history mixes training, evaluation, and final summary rows. Filter by `eval_loss` to inspect validation performance after each epoch.

In [ ]:
training_log_df = pd.DataFrame(trainer.state.log_history)
display(training_log_df)

epoch_eval_df = training_log_df[training_log_df["eval_loss"].notna()].copy()
if len(epoch_eval_df):
    display(epoch_eval_df[["epoch", "step", "eval_loss", "eval_runtime"]])
else:
    print("No eval_loss rows found in trainer log history.")

## Inference

Predictions are generated for the full test set using beam search. Only the `complex` sentence is passed to the model.

In [ ]:
GENERATION_CONFIG = {
    "max_new_tokens": 128,
    "num_beams": 4,
    "length_penalty": 0.9,
    "no_repeat_ngram_size": 3,
    "early_stopping": True,
}

print("Generation config:", GENERATION_CONFIG)

def generate_batch(sentences: list[str]) -> list[str]:
    input_texts = [INPUT_PREFIX + str(sentence) for sentence in sentences]
    inputs = tokenizer(
        input_texts,
        max_length=max_source_length,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **GENERATION_CONFIG,
        )
    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return [text.strip() for text in decoded]


def generate_predictions(df: pd.DataFrame, batch_size: int = 8) -> pd.DataFrame:
    model.eval()
    predictions = []
    sentences = df["complex"].fillna("").astype(str).tolist()

    for start in tqdm(range(0, len(sentences), batch_size), desc="Generating"):
        batch_sentences = sentences[start : start + batch_size]
        try:
            batch_predictions = generate_batch(batch_sentences)
        except Exception as exc:
            print(f"Generation failed for rows {start}-{start + len(batch_sentences) - 1}: {exc}")
            batch_predictions = [""] * len(batch_sentences)
        predictions.extend(batch_predictions)

    output_df = df[["pair_id", "sent_id", "label", "complex", "simple"]].copy()
    output_df["prediction"] = predictions
    return output_df

prediction_df = generate_predictions(test_df, batch_size=8)
prediction_df.to_csv(PREDICTION_PATH, index=False)
print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
display(prediction_df.head())

## Evaluation

The metrics mirror the FLAN-T5 notebook. SARI evaluates simplification edits, BLEU is reported with sacreBLEU on a `0-100` scale for comparability with the E1 result, and BERTScore measures semantic similarity.

In [ ]:
def get_ngrams(tokens: list[str], n: int) -> Counter[tuple[str, ...]]:
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def safe_f1(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def sari_sentence(source: str, prediction: str, reference: str, max_ngram: int = 4) -> float:
    source_tokens = source.lower().split()
    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    add_scores = []
    keep_scores = []
    delete_scores = []

    for n in range(1, max_ngram + 1):
        source_ngrams = set(get_ngrams(source_tokens, n))
        pred_ngrams = set(get_ngrams(pred_tokens, n))
        ref_ngrams = set(get_ngrams(ref_tokens, n))

        add_pred = pred_ngrams - source_ngrams
        add_ref = ref_ngrams - source_ngrams
        add_precision = len(add_pred & add_ref) / len(add_pred) if add_pred else 0.0
        add_recall = len(add_pred & add_ref) / len(add_ref) if add_ref else 0.0
        add_scores.append(safe_f1(add_precision, add_recall))

        keep_pred = pred_ngrams & source_ngrams
        keep_ref = ref_ngrams & source_ngrams
        keep_precision = len(keep_pred & keep_ref) / len(keep_pred) if keep_pred else 0.0
        keep_recall = len(keep_pred & keep_ref) / len(keep_ref) if keep_ref else 0.0
        keep_scores.append(safe_f1(keep_precision, keep_recall))

        delete_pred = source_ngrams - pred_ngrams
        delete_ref = source_ngrams - ref_ngrams
        delete_precision = len(delete_pred & delete_ref) / len(delete_pred) if delete_pred else 0.0
        delete_scores.append(delete_precision)

    return 100 * float(np.mean([np.mean(add_scores), np.mean(keep_scores), np.mean(delete_scores)]))


def compute_sari_score(sources: list[str], predictions: list[str], references: list[str]) -> float:
    if evaluate is not None:
        for metric_name in ["sari", "evaluate-metric/sari"]:
            try:
                sari_metric = evaluate.load(metric_name)
                result = sari_metric.compute(
                    sources=sources,
                    predictions=predictions,
                    references=[[reference] for reference in references],
                )
                return float(result["sari"])
            except Exception:
                pass
    scores = [
        sari_sentence(source, prediction, reference)
        for source, prediction, reference in zip(sources, predictions, references, strict=True)
    ]
    return float(np.mean(scores))


def compute_bleu_score(predictions: list[str], references: list[str]) -> float:
    return float(sacrebleu.corpus_bleu(predictions, [references]).score)


def compute_metrics(df: pd.DataFrame) -> pd.DataFrame:
    valid_df = df.copy()
    valid_df["prediction"] = valid_df["prediction"].fillna("").astype(str)
    valid_df = valid_df[valid_df["prediction"].str.strip().ne("")].reset_index(drop=True)

    sources = valid_df["complex"].tolist()
    predictions = valid_df["prediction"].tolist()
    references = valid_df["simple"].tolist()

    sari_score = compute_sari_score(sources, predictions, references)
    bleu_score = compute_bleu_score(predictions, references)
    bert_precision, bert_recall, bert_f1 = bert_score(
        predictions,
        references,
        lang="en",
        verbose=False,
    )

    return pd.DataFrame(
        [
            {"metric": "SARI", "score": sari_score},
            {"metric": "BLEU", "score": bleu_score},
            {"metric": "BERTScore Precision", "score": float(bert_precision.mean())},
            {"metric": "BERTScore Recall", "score": float(bert_recall.mean())},
            {"metric": "BERTScore F1", "score": float(bert_f1.mean())},
        ]
    )

metrics_summary = compute_metrics(prediction_df)
display(metrics_summary)

## Qualitative Analysis

Inspect random examples to judge whether the model copies the source, removes excessive statistics, replaces biomedical jargon, and preserves meaning.

In [ ]:
example_columns = ["complex", "simple", "prediction"]
qualitative_examples = prediction_df[example_columns].sample(
    n=min(20, len(prediction_df)),
    random_state=SEED,
)
display(qualitative_examples)

### Observation Notes

After reviewing the examples, record observations here:

- Does the model copy the source sentence?
- Does it remove or simplify statistics and technical details?
- Does it replace biomedical jargon with general-audience wording?
- Does it preserve the original meaning?
- How does it compare against FLAN-T5 fine-tuning and Llama zero-shot?

## Final Comparison

This table keeps E0 and E1 baselines beside the computed E2 metrics. BLEU is reported on the sacreBLEU `0-100` scale.

In [ ]:
def metric_value(summary: pd.DataFrame, metric: str) -> float:
    values = summary.loc[summary["metric"].eq(metric), "score"].tolist()
    return float(values[0]) if values else float("nan")

comparison_df = pd.DataFrame(
    [
        {
            "Experiment": "E0",
            "Model": "Llama 3.1 8B zero-shot",
            "SARI": 28.25,
            "BLEU": 3.6,
            "BERTScore F1": 0.897,
        },
        {
            "Experiment": "E1",
            "Model": "FLAN-T5-base fine-tuned",
            "SARI": 26.86,
            "BLEU": 36.57,
            "BERTScore F1": 0.935,
        },
        {
            "Experiment": "E2",
            "Model": f"{RESOLVED_MODEL_NAME} fine-tuned",
            "SARI": metric_value(metrics_summary, "SARI"),
            "BLEU": metric_value(metrics_summary, "BLEU"),
            "BERTScore F1": metric_value(metrics_summary, "BERTScore F1"),
        },
    ]
)
display(comparison_df)

## Documentation Summary

BioBART is used because it starts from a biomedical-domain encoder-decoder checkpoint rather than a general instruction model. Compared with FLAN-T5, this may help the model represent biomedical terminology more accurately while still learning to rewrite sentences into simpler language.

This remains a strict no-context experiment: the only model input is the `complex` sentence. The notebook does not use neighbouring sentences, paragraph context, document context, `doc_pos`, `doc_quint`, or `doc_len`.

This direct BioBART fine-tuning experiment is a step toward the BioBERT + BioBART pipeline described in CLEF-style simplification systems: first establish whether a biomedical generator helps sentence simplification, then later add planning, classification, terminology handling, or context-aware components. This notebook does not implement BioBERT classification, SciSpacy term extraction, plan-guided modelling, or context-aware modelling.